<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

<div style='background-color: #1e3c72; color: white; padding: 20px; border-radius: 10px; text-align: center;'>
<h1>🌪️🛰️ Élaboration de la RGB Dust - MTG / FCI</h1>
<p>Composition d'un produit multispectral pour la détection des poussières</p>
</div>

---

## 🎯 **Objectifs du TP**

- 🔬 Comprendre la recette de la **RGB Dust**
- 📥 Extraire des canaux depuis un fichier **NetCDF**
- 🧮 Calculer des différences spectrales et appliquer des **corrections gamma**
- 🎨 Assembler les bandes en une image **RGB**
- 🗺️ Reprojeter, découper, ajouter des contours et un habillage

---

<div class="alert alert-info">
<b>⚠️ Prérequis</b> : Aucun prérequis technique spécifique, mais des notions de télédétection sont un plus.
</div>

---

<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies nécessaires et configuration des chemins d'accès.
</div>

In [1]:
from PIL import Image
from IPython.display import display, HTML
import time
import matplotlib.pyplot as plt
import numpy as np
import os
import subprocess
from osgeo import gdal

os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'
print("✅ Environnement prêt")

✅ Environnement prêt


## 📂 **Configuration des répertoires**

Définition des chemins d'entrée et de sortie.

In [2]:
cd ~/MF_DATA_MANIPULATION

/home/coster/MF_DATA_MANIPULATION


In [3]:
download_dir = os.path.join(os.getcwd(), "RESULTS")
os.makedirs(download_dir, exist_ok=True)
print(f"📁 Dossier de résultats : {download_dir}")

📁 Dossier de résultats : /home/coster/MF_DATA_MANIPULATION/RESULTS


In [4]:
#❗ Attention : ce chemin doit-être modifié pour toute arborescence différente
input = '../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc'
#input = '../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc'
output = 'RESULTS'
print(f"📥 Fichier source : {input}")

📥 Fichier source : ../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc


### 🔍 **Aperçu général du fichier NetCDF**

La commande `gdalinfo` permet de visualiser la structure et les sous-datasets disponibles.

In [5]:
!gdalinfo {input}

ERROR 4: ../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc: No such file or directory
gdalinfo failed - unable to open '../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc'.


#### 🌡️ **Informations sur le canal IR_105 (10.5 µm)**

Affichage des métadonnées spécifiques à la bande infrarouge 10.5 µm.

In [6]:
!gdalinfo -mm NETCDF:"{input}":IR_105

ERROR 4: NETCDF:../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc:IR_105: No such file or directory
gdalinfo failed - unable to open 'NETCDF:../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc:IR_105'.


## 📜 **Recette de la RGB Dust**

La recette du produit Cloud Type RGB est la suivante :

<img src='../OUTILS/dust_tableau.png' align='left' width='50%'/>

| Composante | Canaux | Étendue | Gamma |
|------------|--------|---------|-------|
| 🔴 **Red** | IR12.3 – IR10.5 | -7,1 à +2,4 K | 1.0 |
| 🟢 **Green** | IR10.5 – IR8.7 | 0,2 à +12,7 K | 2.5 |
| 🔵 **Blue** | IR10.5 | 260,9 à 289 K | 1.0 |

> 💡 <a href="../DOCS/RGB-WS-2025_MeetingReport_final.pdf" target="_blank">Lien vers les recettes RGB du Workshop</a> </br>
> 💡 <a href="https://nextcloud.meteo.fr/s/GDZtnLSsKtKgCwP" target="_blank">Lien vers la fiche</a> 

## 📥 **Extraction des canaux nécessaires**

Conversion des bandes IR_105, IR_123 et IR_087 en fichiers GeoTIFF.

In [7]:
!gdal_translate NETCDF:"$input":IR_105 {output}/105.tif >/dev/null 2>&1
!gdal_translate NETCDF:"$input":IR_123 {output}/123.tif >/dev/null 2>&1
!gdal_translate NETCDF:"$input":IR_087 {output}/087.tif >/dev/null 2>&1

## 🔵 **Préparation de la bande Bleue (IR10.5)**

Les données sont stockées en centièmes de degré Celsius. 

In [8]:
!gdalinfo -mm NETCDF:"{input}":IR_105 | grep Max 

ERROR 4: NETCDF:../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc:IR_105: No such file or directory
gdalinfo failed - unable to open 'NETCDF:../stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301200.nc:IR_105'.


Étirement linéaire entre 260,9 K (-12,25 °C) et 289 K (15,85 °C).

In [9]:
!gdal_translate -ot byte -scale -1215 1585 0 255 NETCDF:"$input":IR_105 {output}/blue_105_-1215_1585.tif 2>/dev/null

## 🔴🟢  **Calcul des différences spectrales**

- Rouge : `IR_123 - IR_105`
- Vert : `IR_105 - IR_087`

In [10]:
!gdal_calc.py -A {output}/123.tif -B {output}/105.tif --outfile={output}/red_123_105.tif --calc="A-B" 2>/dev/null
!gdal_calc.py -A {output}/105.tif -B {output}/087.tif --outfile={output}/green_105_087.tif --calc="A-B" 2>/dev/null

### 🎚️ **Mise à l'échelle des différences, et ajout de la correction gamma sur la bande verte**

| Bande | Étendue (K) | Commande |
|-------|-------------|----------|
| Rouge | -7,1 à +2,4 K | `-scale -710 240 0 255` |
| Vert  | 0,2 à +12,7 K | `-scale 020 1270 0 255` |

In [11]:
#Bande rouge
!gdal_translate -ot byte -scale -710 240 0 255 {output}/red_123_105.tif {output}/red_123_105_scale.tif 2>/dev/null

In [12]:
#Bande verte
gamma=2.5
Valgamma=1/gamma
!gdal_translate -ot byte -scale 20 1270 0 255 -exponent {Valgamma} {output}/green_105_087.tif {output}/green_105_087_scale_gamma.tif

ERROR 4: RESULTS/green_105_087.tif: No such file or directory


## 🖼️ **Assemblage de l'image RGB**

Fusion des trois bandes (Rouge, Vert, Bleu) en une seule image GeoTIFF.

In [13]:
!gdal_merge.py -separate {output}/red_123_105_scale.tif {output}/green_105_087_scale_gamma.tif {output}/blue_105_-1215_1585.tif -o {output}/RGB_dust.tif
!convert -resize 1000x1000 {output}/RGB_dust.tif {output}/RGB_dust.jpg
display(Image.open(output + '/RGB_dust.jpg'))

Traceback (most recent call last):
  File "/home/coster/.conda/envs/env_MF_teledetection/bin/gdal_merge.py", line 12, in <module>
    sys.exit(main(sys.argv))
  File "/home/coster/.conda/envs/env_MF_teledetection/lib/python3.10/site-packages/osgeo_utils/gdal_merge.py", line 657, in main
    return gdal_merge(argv)
  File "/home/coster/.conda/envs/env_MF_teledetection/lib/python3.10/site-packages/osgeo_utils/gdal_merge.py", line 397, in gdal_merge
    return _gdal_merge(argv=argv)
  File "/home/coster/.conda/envs/env_MF_teledetection/lib/python3.10/site-packages/osgeo_utils/gdal_merge.py", line 530, in _gdal_merge
    file_infos = names_to_fileinfos(names)
  File "/home/coster/.conda/envs/env_MF_teledetection/lib/python3.10/site-packages/osgeo_utils/gdal_merge.py", line 213, in names_to_fileinfos
    if fi.init_from_name(name) == 1:
  File "/home/coster/.conda/envs/env_MF_teledetection/lib/python3.10/site-packages/osgeo_utils/gdal_merge.py", line 247, in init_from_name
    fh = gdal.Ope

FileNotFoundError: [Errno 2] No such file or directory: 'RESULTS/RGB_dust.jpg'

## 🗺️ **Découpage et reprojection (EPSG:4326)**

On extrait la zone centrée sur l'Afrique de l'Ouest et les îles du Cap-Vert.

In [ ]:
!gdalwarp -overwrite -t_srs "EPSG:4326" -te -19 20 10 37 {output}/RGB_dust.tif {output}/RGB_dust_decoupe.tif

## 🧭 **Ajout des contours administratifs**

Utilisation de `gdal_rasterize` pour dessiner les frontières en blanc.

In [ ]:
!gdal_rasterize -q -b 1 -burn 255 -b 2 -burn 255 -b 3 -burn 255 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp {output}/RGB_dust_decoupe.tif
!convert -resize 1400x1400 {output}/RGB_dust_decoupe.tif {output}/RGB_dust_decoupe.jpg
display(Image.open(output + '/RGB_dust_decoupe.jpg'))

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice</div>

- Produire Dust RGB pour les heures suivantes :</br>
Fichiers NetCDF disponibles : du 30/03/2025 à 12 h UTC au 31/03/2026 à 12 h UTC, pas de temps horaire</br>
Répertoire : /stockage/DATA/202603/</br>
(exemple : '/stockage/DATA/202603/Mmultic2kmNC4_mtgi1_202603301300.nc'



- Découper les TIFS Geocolor RGB sur la même zone, pour les mêmes heures que Dust RGB</br>
Fichiers Tiff disponibles : du 30/03/2025 à 12 h UTC au 31/03/2026 à 12 h UTC, pas de temps horaire</br>
répertoire : ../stockage/DATA/202603/Geocolor_rgb
(exemple : '../stockage/DATA/202603/Geocolor_rgb/202603301300.geotiff')</br>
ou répertoire : /stockage/DATA/202603/Geocolor_rgb
(exemple : '/stockage/DATA/202603/Geocolor_rgb/202603301300.geotiff')



> 💡 Découper & reprojeter

✅ Fin du TP